# Full-res U-Net TRAINING

**Recommended — from-cache mode** (cell 4): attach the four cache datasets
(`climx-cache-ssp126`, `-ssp370`, `-ssp585`, `-extras`) as inputs; they are
partially unpacked to /tmp (one scenario + hist, ~35 GB; X_test read from the mount) and then fit + evaluate run with NO slow reads and
NO re-preprocessing.

**Fallback — one-shot** (cell 6): no datasets attached; the script downloads the
raw files from Hugging Face itself (~62 MB/s) and builds a ×3 cache in-session.

**Setup:** right panel → **Data → Add Input → Datasets** → add the cache datasets
(or nothing, for the fallback) → **Accelerator → GPU (T4 x2)**.

Ends with `FULLRES_TRAIN_OK` and the submission parquet path.

## 1. Clone the repo + install deps (~2 min, skips on reruns)

In [ ]:
import os, subprocess, sys, pathlib
repo = "/kaggle/working/climx"
if os.path.isdir(repo):
    r = subprocess.run(["git", "-C", repo, "pull", "--ff-only"], capture_output=True, text=True)
    print(r.stdout or r.stderr)
else:
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/nonewanit/climx-emulator.git", repo], check=True)
os.chdir(repo)
sys.path.insert(0, str(pathlib.Path(repo) / "src"))
subprocess.run(["find", repo, "-name", "__pycache__", "-type", "d",
                "-exec", "rm", "-rf", "{}", "+"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"], check=True)
r = subprocess.run(["git", "-C", repo, "log", "-1", "--oneline"], capture_output=True, text=True)
print("HEAD:", r.stdout.strip(), "— MUST be the commit None told you")
print("setup done")

## 2. Train from the uploaded cache (RECOMMENDED)

Attach ALL FOUR datasets as inputs first: `climx-cache-ssp126`, `climx-cache-ssp370`,
`climx-cache-ssp585`, `climx-cache-extras` (they must be shared with this account).
Expected: copy ~2 h → 2 epochs ~2 h → evaluate ~2.5 h.

In [ ]:
# Session storage quota is ~57.6 GiB - the full x1 cache doesn't fit in /tmp.
# Train on ONE scenario (its stores ~24 GB + hist ~11 GB fit); X_test.nc is
# read directly from the mount during predict. Change ssp370 to ssp126/ssp585
# for later runs.
%run scripts/fullres_train.py --scenarios ssp370 --epochs 2 --batch 16

## 3. Fallback: ONE-SHOT (no datasets; downloads raw from HF, ×3 cache)

In [ ]:
%run scripts/fullres_train.py --subsample 3 --epochs 2 --batch 16

## 4. Submit the parquet (2/day team quota — coordinate!)

```bash
!kaggle competitions submit -c climx-main-track -f results/submissions/submission_unet_full.parquet -m "U-Net full-res"
```
Or download it (right panel → Output) and upload on the competition page.